# Wide-Format Ingest & Export

Tests for the canonical wide column convention and the `BinauralAudiogram.to_wide_row()` / `from_wide_row()` round-trip.

Covers: column naming, parsing, round-trip, column map from foreign datasets, sparse data, edge cases.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

from audiogram_object import (
    ThresholdPoint, EarAudiogram, BinauralAudiogram,
    wide_column_name, parse_wide_column, canonical_wide_columns, apply_column_map,
    register_summary_metric, unregister_summary_metric, SUMMARY_METRICS,
)

## Column Naming Convention

Pattern: `{ear}_{pathway}_{frequency}` with optional `_masked` / `_nr` suffixes.

In [ ]:
# Build canonical names
assert wide_column_name('right', 'air', 500) == 'r_air_500'
assert wide_column_name('left', 'bone', 1000) == 'l_bone_1000'
assert wide_column_name('right', 'air', 4000, 'masked') == 'r_air_4000_masked'
assert wide_column_name('left', 'air', 250, 'nr') == 'l_air_250_nr'

print('wide_column_name examples:')
for ear, pw, f in [('right','air',500), ('left','bone',2000), ('right','air',8000)]:
    print(f'  {ear}, {pw}, {f} -> {wide_column_name(ear, pw, f)}')

print('All assertions passed.')

In [ ]:
# Parse canonical names back to components
assert parse_wide_column('r_air_500') == {'ear': 'right', 'pathway': 'air', 'freq_hz': 500, 'field': 'threshold'}
assert parse_wide_column('l_bone_1000_masked') == {'ear': 'left', 'pathway': 'bone', 'freq_hz': 1000, 'field': 'masked'}
assert parse_wide_column('r_air_4000_nr') == {'ear': 'right', 'pathway': 'air', 'freq_hz': 4000, 'field': 'nr'}

# Non-audiometric columns return None
assert parse_wide_column('patient_id') is None
assert parse_wide_column('audiogram_id') is None
assert parse_wide_column('garbage_column') is None
assert parse_wide_column('x_air_500') is None  # invalid ear
assert parse_wide_column('r_speech_500') is None  # invalid pathway

print('All parse assertions passed.')

In [ ]:
# Generate canonical column lists
print('Air only, standard clinical frequencies:')
cols = canonical_wide_columns([250, 500, 1000, 2000, 4000, 8000])
print(cols)
assert len(cols) == 12  # 6 freqs x 2 ears

print()
print('Air + bone with masked flags:')
cols_full = canonical_wide_columns(
    [500, 1000, 2000],
    pathways=['air', 'bone'],
    include_masked=True,
)
print(cols_full)
assert len(cols_full) == 24  # 3 freqs x 2 pathways x 2 ears x 2 (threshold + masked)

print()
print('Default (all standard frequencies, air only):')
cols_default = canonical_wide_columns()
print(f'{len(cols_default)} columns: {cols_default[:4]} ... {cols_default[-4:]}')

## Round-Trip: to_wide_row / from_wide_row

Full clinical audiogram with air, bone, masked, and NR — export and reimport.

In [ ]:
ba = BinauralAudiogram(
    left=EarAudiogram(
        air={
            250: 10.0, 500: 15.0,
            1000: ThresholdPoint(30.0, masked=True),
            2000: 40.0,
            4000: ThresholdPoint(120.0, nr=True),
        },
        bone={
            500: ThresholdPoint(10.0, masked=True),
            1000: ThresholdPoint(20.0, masked=True),
        },
    ),
    right=EarAudiogram(
        air={250: 15.0, 500: 25.0, 1000: 35.0, 2000: 45.0},
    ),
    audiogram_id='wide-test-001',
    subject_id='pt-42',
    performed_at='2024-06-15',
    source='clinic',
)

wide = ba.to_wide_row()
print('Exported wide row:')
for k, v in wide.items():
    print(f'  {k}: {v}')

In [ ]:
ba2 = BinauralAudiogram.from_wide_row(wide)

assert ba == ba2, 'Round-trip equality failed!'
print('Round-trip: PASSED')
print()
print('Left air:', {f: (p.threshold_db, p.masked, p.nr) for f, p in sorted(ba2.left.air.items())})
print('Left bone:', {f: (p.threshold_db, p.masked, p.nr) for f, p in sorted(ba2.left.bone.items())})
print('Right air:', {f: (p.threshold_db, p.masked, p.nr) for f, p in sorted(ba2.right.air.items())})
print(f'Meta: id={ba2.audiogram_id}, subj={ba2.subject_id}, date={ba2.performed_at}, src={ba2.source}')

## Column Map: Foreign Dataset Ingest

Simulate importing from a research dataset with non-canonical column names.

The user writes a `column_map` dict once, then every row in the dataset can be ingested.

In [ ]:
# A row from a hypothetical research export
research_row = {
    'PatientID': 'PT-099',
    'TestDate': '2024-01-15',
    'Facility': 'MEI',
    'R_AC_250': 20, 'R_AC_500': 30, 'R_AC_1K': 40, 'R_AC_2K': 55, 'R_AC_4K': 65, 'R_AC_8K': 70,
    'L_AC_250': 15, 'L_AC_500': 20, 'L_AC_1K': 25, 'L_AC_2K': 35, 'L_AC_4K': 40, 'L_AC_8K': 50,
    'R_BC_500': 25, 'R_BC_1K': 35, 'R_BC_2K': 50,
    'L_BC_500': 15, 'L_BC_1K': 20, 'L_BC_2K': 30,
}

# One-time column map for this dataset
col_map = {
    'PatientID': 'subject_id',
    'TestDate': 'performed_at',
    'Facility': 'source',
    'R_AC_250':  'r_air_250',  'R_AC_500':  'r_air_500',
    'R_AC_1K':   'r_air_1000', 'R_AC_2K':   'r_air_2000',
    'R_AC_4K':   'r_air_4000', 'R_AC_8K':   'r_air_8000',
    'L_AC_250':  'l_air_250',  'L_AC_500':  'l_air_500',
    'L_AC_1K':   'l_air_1000', 'L_AC_2K':   'l_air_2000',
    'L_AC_4K':   'l_air_4000', 'L_AC_8K':   'l_air_8000',
    'R_BC_500':  'r_bone_500', 'R_BC_1K':   'r_bone_1000', 'R_BC_2K': 'r_bone_2000',
    'L_BC_500':  'l_bone_500', 'L_BC_1K':   'l_bone_1000', 'L_BC_2K': 'l_bone_2000',
}

ba_research = BinauralAudiogram.from_wide_row(research_row, column_map=col_map)

print(f'Subject: {ba_research.subject_id}')
print(f'Date: {ba_research.performed_at}')
print(f'Source: {ba_research.source}')
print(f'Right PTA: {ba_research.right.pta():.1f}')
print(f'Left PTA: {ba_research.left.pta():.1f}')
print(f'Asymmetric (any_15db): {ba_research.is_asymmetric("any_15db")}')
print(f'Right bone: {sorted((f, p.threshold_db) for f, p in ba_research.right.bone.items())}')
print(f'Left bone: {sorted((f, p.threshold_db) for f, p in ba_research.left.bone.items())}')

## apply_column_map utility

Low-level helper — renames keys in a dict. Unmapped keys pass through.

In [ ]:
raw = {'R_AC_500': 25.0, 'patient': 'A', 'extra_col': 99}
mapped = apply_column_map(raw, {'R_AC_500': 'r_air_500'})
print('Before:', raw)
print('After: ', mapped)
assert mapped == {'r_air_500': 25.0, 'patient': 'A', 'extra_col': 99}
print('Assertion passed — unmapped keys preserved.')

## Sparse Data

Most research datasets have only air thresholds, no bone, no masked/NR flags.

Verify that missing columns default gracefully.

In [ ]:
sparse_row = {
    'r_air_500': 20, 'r_air_1000': 30, 'r_air_2000': 40,
    'l_air_500': 15, 'l_air_1000': 25, 'l_air_2000': 35,
}

ba_sparse = BinauralAudiogram.from_wide_row(sparse_row)

print('Right air:', {f: p.threshold_db for f, p in sorted(ba_sparse.right.air.items())})
print('Left air:', {f: p.threshold_db for f, p in sorted(ba_sparse.left.air.items())})
print('Right bone:', ba_sparse.right.bone)  # should be empty
print('Left bone:', ba_sparse.left.bone)    # should be empty

# All flags should be False
for ear in [ba_sparse.left, ba_sparse.right]:
    for f, p in ear.air.items():
        assert not p.masked and not p.nr, f'Unexpected flag at {f} Hz'

print('All flags default to False — PASSED')

## Missing / Empty Values

Wide rows from real datasets often have `None`, `NaN`, or empty strings for untested frequencies.

These should be silently skipped.

In [ ]:
import math

messy_row = {
    'r_air_250': 20,
    'r_air_500': 30,
    'r_air_1000': None,       # not tested
    'r_air_2000': '',         # empty string
    'r_air_4000': 50,
    'l_air_500': 15,
    'l_air_1000': 25,
}

ba_messy = BinauralAudiogram.from_wide_row(messy_row)

print('Right air freqs:', sorted(ba_messy.right.air.keys()))
assert 1000 not in ba_messy.right.air, '1000 Hz (None) should be skipped'
assert 2000 not in ba_messy.right.air, '2000 Hz (empty string) should be skipped'
assert sorted(ba_messy.right.air.keys()) == [250, 500, 4000]
print('None and empty string values skipped — PASSED')

## Batch Ingest from DataFrame

Typical workflow: user has a pandas DataFrame with one row per audiogram.

In [ ]:
# Simulate a small DataFrame as list of dicts
dataset = [
    {'id': 'A001', 'R_AC_500': 20, 'R_AC_1K': 30, 'R_AC_2K': 40,
     'L_AC_500': 15, 'L_AC_1K': 20, 'L_AC_2K': 25},
    {'id': 'A002', 'R_AC_500': 40, 'R_AC_1K': 55, 'R_AC_2K': 60,
     'L_AC_500': 35, 'L_AC_1K': 45, 'L_AC_2K': 50},
    {'id': 'A003', 'R_AC_500': 10, 'R_AC_1K': 10, 'R_AC_2K': 15,
     'L_AC_500': 10, 'L_AC_1K': 15, 'L_AC_2K': 15},
]

# Define column map once for this dataset
col_map = {
    'id': 'audiogram_id',
    'R_AC_500': 'r_air_500', 'R_AC_1K': 'r_air_1000', 'R_AC_2K': 'r_air_2000',
    'L_AC_500': 'l_air_500', 'L_AC_1K': 'l_air_1000', 'L_AC_2K': 'l_air_2000',
}

# Ingest all rows
audiograms = [BinauralAudiogram.from_wide_row(row, column_map=col_map) for row in dataset]

for a in audiograms:
    ptas = a.pta()
    print(f'{a.audiogram_id}: R PTA={ptas["right"]:.0f}, L PTA={ptas["left"]:.0f}')

print(f'\nIngested {len(audiograms)} audiograms successfully.')

## Metadata Overrides

`from_wide_row` accepts keyword arguments to override metadata instead of pulling from the row.

In [ ]:
bare_row = {'r_air_500': 30, 'r_air_1000': 40, 'l_air_500': 25, 'l_air_1000': 30}

ba_override = BinauralAudiogram.from_wide_row(
    bare_row,
    audiogram_id='manual-001',
    subject_id='pt-100',
    performed_at='2025-03-01',
    source='research',
)

assert ba_override.audiogram_id == 'manual-001'
assert ba_override.subject_id == 'pt-100'
assert ba_override.performed_at == '2025-03-01'
assert ba_override.source == 'research'
print(ba_override)
print('Metadata override — PASSED')

## Export Without Metadata

`to_wide_row(include_meta=False)` omits the metadata columns.

In [ ]:
wide_no_meta = ba.to_wide_row(include_meta=False)
print('Keys:', sorted(wide_no_meta.keys()))
assert 'audiogram_id' not in wide_no_meta
assert 'subject_id' not in wide_no_meta
print('No metadata columns — PASSED')

## summary() — Computed Metrics as a Flat Dict

`BinauralAudiogram.summary()` returns all registered metrics in a single dict, ready to merge into a DataFrame row.

Built-in metric categories: `pta`, `asymmetry`, `frequency_count`.

In [ ]:
ba_summary = BinauralAudiogram(
    left=EarAudiogram(air={500: 45, 1000: 50, 2000: 55}),
    right=EarAudiogram(air={500: 20, 1000: 25, 2000: 30}),
)

print('=== All metrics ===')
for k, v in ba_summary.summary().items():
    print(f'  {k}: {v}')

In [ ]:
# Filter by category
print('=== include=["pta"] ===')
for k, v in ba_summary.summary(include=['pta']).items():
    print(f'  {k}: {v}')

print()
print('=== exclude=["frequency_count"] ===')
for k, v in ba_summary.summary(exclude=['frequency_count']).items():
    print(f'  {k}: {v}')

## Custom Metrics

Register your own metric function. It receives a `BinauralAudiogram` and returns a flat dict.

In [ ]:
def hearing_loss_degree(ba):
    """Classify each ear by WHO hearing loss grade based on PTA."""
    grades = [
        (25, 'normal'), (40, 'mild'), (55, 'moderate'),
        (70, 'moderately_severe'), (90, 'severe'), (float('inf'), 'profound'),
    ]
    result = {}
    for ear_name in ('right', 'left'):
        pta = ba.pta()[ear_name]
        label = None
        if pta is not None:
            for cutoff, grade in grades:
                if pta <= cutoff:
                    label = grade
                    break
        result[f'{ear_name}_hl_degree'] = label
    return result

register_summary_metric('hl_degree', hearing_loss_degree)

print('=== With custom metric ===')
for k, v in ba_summary.summary().items():
    print(f'  {k}: {v}')

# Clean up
unregister_summary_metric('hl_degree')
assert 'hl_degree' not in SUMMARY_METRICS
print('\nCustom metric registered, used, and unregistered — PASSED')

## Enrichment Workflow

The primary use case: ingest a wide dataset, compute metrics, append back to the original data.

In [ ]:
# Simulate a research dataset (list of dicts = DataFrame rows)
dataset = [
    {'id': 'A001', 'age': 55, 'R_AC_500': 20, 'R_AC_1K': 30, 'R_AC_2K': 40,
     'L_AC_500': 15, 'L_AC_1K': 20, 'L_AC_2K': 25},
    {'id': 'A002', 'age': 72, 'R_AC_500': 40, 'R_AC_1K': 55, 'R_AC_2K': 60,
     'L_AC_500': 35, 'L_AC_1K': 45, 'L_AC_2K': 50},
    {'id': 'A003', 'age': 34, 'R_AC_500': 10, 'R_AC_1K': 10, 'R_AC_2K': 15,
     'L_AC_500': 10, 'L_AC_1K': 15, 'L_AC_2K': 15},
]

col_map = {
    'id': 'audiogram_id',
    'R_AC_500': 'r_air_500', 'R_AC_1K': 'r_air_1000', 'R_AC_2K': 'r_air_2000',
    'L_AC_500': 'l_air_500', 'L_AC_1K': 'l_air_1000', 'L_AC_2K': 'l_air_2000',
}

# Enrich each row with computed metrics
enriched = []
for row in dataset:
    ba = BinauralAudiogram.from_wide_row(row, column_map=col_map)
    enriched_row = {**row, **ba.summary(include=['pta', 'asymmetry'])}
    enriched.append(enriched_row)

print(f'{"id":<6} {"age":>3}  {"R PTA":>6} {"L PTA":>6} {"better":>7} {"asym_15db"}')
print('-' * 55)
for r in enriched:
    print(f'{r["id"]:<6} {r["age"]:>3}  '
          f'{r["pta_right"]:>6.1f} {r["pta_left"]:>6.1f} '
          f'{r["better_ear_pta"]:>7.1f} {str(r["asymmetric_any_15db"]):>9}')

print(f'\nOriginal columns preserved: {[k for k in enriched[0] if k in dataset[0]]}')

## Summary

Run all cells above — all assertions should pass with no errors.

In [ ]:
print('All wide-format tests passed.')